In [ ]:
"""
A script to fetch the loci in a search query from the ANTARES API.
"""

#from antares_client.search import search

QUERY = {
    "query": {
        "bool": {
            "filter": {
                "bool": {
                    "must": [
                        {
                            "terms": {
                                "tags": [
                                    "lantern_xgboost_t2.0.7_c0.95"
                                ]
                            }
                        },
                        {
                            "exists": {
                                "field": "properties.survey.lsst"
                            }
                        }
                    ]
                }
            }
        }
    }
}


#def process_locus(locus):
#    """Put your custom processing logic here."""
#    print(locus.locus_id)


#def main():
#    for locus in search(QUERY):
#        process_locus(locus)


#if __name__ == "__main__":
#    main()



In [ ]:
import json
import pandas as pd
from pathlib import Path
from antares_client.search import search

In [ ]:
%%time


# ============================================================
# Settings
# ============================================================

outdir = Path("antares_data")

locus_dir = outdir / "loci"
alert_dir = outdir / "alerts"

locus_dir.mkdir(parents=True, exist_ok=True)
alert_dir.mkdir(parents=True, exist_ok=True)

# Independent batch sizes
alert_batch_size = 100_000
locus_batch_size = 5_000


# ============================================================
# Temporary storage
# ============================================================

alert_rows = []
locus_rows = []

alert_part = 0
locus_part = 0

n_alerts_total = 0
n_alerts_saved = 0
n_ztf_skipped = 0


# ============================================================
# Search ANTARES
# ============================================================

for n_loci, locus in enumerate(search(QUERY), start=1):

    if n_loci%25==0: print(f'n_loci: {n_loci}')

    # --------------------------------------------------------
    # Locus-level information
    # --------------------------------------------------------

    locus_rows.append({
        "locus_id": locus.locus_id,
        "ra": locus.ra,
        "dec": locus.dec,
        "tags": locus.tags,
        "locus_properties": json.dumps(
            locus.properties,
            separators=(",", ":"),
            default=str,
        ),
    })


    # --------------------------------------------------------
    # Alert-level information
    # --------------------------------------------------------

    for alert in locus.alerts:

        n_alerts_total += 1

        # Skip ZTF alerts
        if not alert.alert_id.startswith("lsst:"):
            n_ztf_skipped += 1
            continue

        alert_rows.append({
            "locus_id": locus.locus_id,
            "alert_id": alert.alert_id,
            "mjd": alert.mjd,
            "alert_properties": json.dumps(
                alert.properties,
                separators=(",", ":"),
                default=str,
            ),
        })

        n_alerts_saved += 1


        # ----------------------------------------------------
        # Flush alerts independently
        # ----------------------------------------------------

        if len(alert_rows) >= alert_batch_size:

            pd.DataFrame(alert_rows).to_parquet(
                alert_dir / f"alerts_{alert_part:05d}.parquet",
                index=False,
                compression="zstd",
            )

            print(
                f"Saved alert part {alert_part:05d}: "
                f"{len(alert_rows):,} alerts"
            )

            alert_rows = []
            alert_part += 1


    # --------------------------------------------------------
    # Flush loci independently
    # --------------------------------------------------------

    if len(locus_rows) >= locus_batch_size:

        pd.DataFrame(locus_rows).to_parquet(
            locus_dir / f"loci_{locus_part:05d}.parquet",
            index=False,
            compression="zstd",
        )

        print(
            f"Saved locus part {locus_part:05d}: "
            f"{len(locus_rows):,} loci "
            f"(processed {n_loci:,} total)"
        )

        locus_rows = []
        locus_part += 1


    if n_loci==1_000: break

# ============================================================
# Save remaining alerts
# ============================================================

if alert_rows:

    pd.DataFrame(alert_rows).to_parquet(
        alert_dir / f"alerts_{alert_part:05d}.parquet",
        index=False,
        compression="zstd",
    )

    print(
        f"Saved final alert part {alert_part:05d}: "
        f"{len(alert_rows):,} alerts"
    )


# ============================================================
# Save remaining loci
# ============================================================

if locus_rows:

    pd.DataFrame(locus_rows).to_parquet(
        locus_dir / f"loci_{locus_part:05d}.parquet",
        index=False,
        compression="zstd",
    )

    print(
        f"Saved final locus part {locus_part:05d}: "
        f"{len(locus_rows):,} loci"
    )


# ============================================================
# Summary
# ============================================================

print()
print("Finished")
print(f"Loci processed:          {n_loci:,}")
print(f"All alerts encountered:  {n_alerts_total:,}")
print(f"Non-ZTF alerts saved:    {n_alerts_saved:,}")
print(f"ZTF alerts skipped:      {n_ztf_skipped:,}")

In [ ]:
import pyarrow.dataset as ds

loci_ds = ds.dataset(
    "antares_data/loci",
    format="parquet",
)

alerts_ds = ds.dataset(
    "antares_data/alerts",
    format="parquet",
)

In [ ]:
locus_id = "ANT2020etnce"

locus = loci_ds.to_table(
    filter=ds.field("locus_id") == locus_id
).to_pandas()

alerts = alerts_ds.to_table(
    filter=ds.field("locus_id") == locus_id
).to_pandas()

In [ ]:
locus

In [ ]:
alerts

In [ ]:
l = alerts['alert_id'].to_list()
l.sort()
print(l[:5])

In [ ]:
coord = loci_ds.to_table(
    columns=["locus_id", "ra", "dec"],
    filter=ds.field("locus_id") == locus_id,
).to_pandas()

In [ ]:
coord